In [8]:
#!/usr/bin/env python3
"""
Conceptual Quantum Circuit Diagrams
NEQR + Code 1 Secret Sharing + Quantum Parity Authentication (QPA)
===================================================================
Pure matplotlib — no Qiskit needed. Runs anywhere.

Diagrams:
  C1. NEQR Position Superposition
  C2. NEQR Single Pixel Encoding
  C3. Share Generation S1, S2, S3 (Code 1 nibble XOR)
  C4. QPA — Quantum Parity Authentication
  C5. Full Encoding Pipeline (all stages combined)
  C6. Decoding — Reverse CNOT + QPA Verification
  MASTER. Full scheme one-page overview (paper figure)
"""

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch, Circle
import numpy as np
import os

OUT_DIR = "conceptual_circuits"
os.makedirs(OUT_DIR, exist_ok=True)

# ── Palette ──────────────────────────────────────────────────────────────────
BG    = '#0d1117'
PANEL = '#161b22'
WIRE  = '#58a6ff'
H_C   = '#1f4068'
X_C   = '#3d1f68'
CNOT_C= '#1f6feb'
MCX_C = '#0d419d'
QPA_C = '#6e40c9'
MEAS_C = '#196c2e'
CTRL  = '#f85149'
TEXT  = '#e6edf3'
DIM   = '#8b949e'
GRN   = '#3fb950'
RED   = '#f85149'
GOLD  = '#d29922'
TEAL  = '#39d353'
BLUE  = '#58a6ff'
PRP   = '#bc8cff'
ORG   = '#db6d28'


# ─────────────────────────────────────────────────────────────────────────────
# PRIMITIVES
# ─────────────────────────────────────────────────────────────────────────────

def wire(ax, row, x0, x1, color=WIRE, lw=1.5):
    ax.plot([x0, x1], [row, row], color=color, lw=lw, zorder=1, solid_capstyle='round')

def cwire(ax, row, x0, x1):
    ax.plot([x0, x1], [row+0.04, row+0.04], color=GOLD, lw=1.0, zorder=1)
    ax.plot([x0, x1], [row-0.04, row-0.04], color=GOLD, lw=1.0, zorder=1)

def gate(ax, x, row, label, w=0.50, h=0.36, fc=CNOT_C, tc=TEXT, fs=9, bold=True):
    r = FancyBboxPatch((x-w/2, row-h/2), w, h,
                       boxstyle='round,pad=0.03',
                       facecolor=fc, edgecolor=WIRE, lw=1.3, zorder=3)
    ax.add_patch(r)
    ax.text(x, row, label, ha='center', va='center', color=tc,
            fontsize=fs, fontweight='bold' if bold else 'normal',
            fontfamily='monospace', zorder=4)

def H(ax, x, row):  gate(ax, x, row, 'H',  fc=H_C, tc=TEAL, fs=11)
def Xg(ax, x, row): gate(ax, x, row, 'X',  fc=X_C, tc=PRP,  fs=11)

def CNOT(ax, x, c_row, t_row, color=CNOT_C):
    lo, hi = min(c_row, t_row), max(c_row, t_row)
    ax.plot([x, x], [lo, hi], color=color, lw=1.5, zorder=2)
    ax.add_patch(Circle((x, c_row), 0.07, fc=CTRL, ec=CTRL, zorder=5))
    ax.add_patch(Circle((x, t_row), 0.17, fc='none', ec=color, lw=1.8, zorder=4))
    ax.plot([x-0.17, x+0.17], [t_row, t_row], color=color, lw=1.5, zorder=5)
    ax.plot([x, x], [t_row-0.17, t_row+0.17], color=color, lw=1.5, zorder=5)

def MCX(ax, x, ctrl_rows, t_row, color=MCX_C):
    all_rows = list(ctrl_rows) + [t_row]
    ax.plot([x, x], [min(all_rows), max(all_rows)], color=color, lw=1.5, zorder=2)
    for cr in ctrl_rows:
        ax.add_patch(Circle((x, cr), 0.07, fc=CTRL, ec=CTRL, zorder=5))
    ax.add_patch(Circle((x, t_row), 0.17, fc='none', ec=color, lw=1.8, zorder=4))
    ax.plot([x-0.17, x+0.17], [t_row, t_row], color=color, lw=1.5, zorder=5)
    ax.plot([x, x], [t_row-0.17, t_row+0.17], color=color, lw=1.5, zorder=5)

def MEAS(ax, x, row):
    r = FancyBboxPatch((x-0.26, row-0.18), 0.52, 0.36,
                       boxstyle='round,pad=0.02',
                       facecolor=MEAS_C, edgecolor=GRN, lw=1.3, zorder=3)
    ax.add_patch(r)
    th = np.linspace(np.pi, 0, 30)
    ax.plot(x+0.13*np.cos(th), row-0.03+0.10*np.sin(th), color=TEXT, lw=1.1, zorder=5)
    ax.plot([x, x+0.11], [row-0.03, row+0.08], color=TEXT, lw=1.1, zorder=5)

draw_meas = MEAS

def barrier(ax, x, r0, r1, pad=0.3):
    ax.plot([x, x], [r0-pad, r1+pad], color=DIM, lw=0.9, ls='--', zorder=2, alpha=0.55)

def rlabel(ax, row, text, x=-0.3, color=TEXT):
    ax.text(x, row, text, ha='right', va='center', color=color,
            fontsize=8, fontfamily='monospace')

def blabel(ax, x, y, text, color=GOLD, fs=7):
    ax.text(x, y, text, ha='center', va='top', color=color,
            fontsize=fs, fontfamily='sans-serif')

def make_ax(nrows, width, title, subtitle='', fig_h=None):
    fw = max(9, width)
    fh = fig_h or max(3.5, nrows*0.80 + 1.6)
    fig, ax = plt.subplots(figsize=(fw, fh))
    fig.patch.set_facecolor(BG)
    ax.set_facecolor(PANEL)
    ax.set_xlim(-0.6, width)
    ax.set_ylim(nrows - 0.5, -0.8)
    ax.axis('off')
    ax.text(0.5, 1.04, title, transform=ax.transAxes,
            ha='center', va='bottom', color=TEXT, fontsize=11, fontweight='bold',
            fontfamily='sans-serif')
    if subtitle:
        ax.text(0.5, 1.00, subtitle, transform=ax.transAxes,
                ha='center', va='bottom', color=DIM, fontsize=8,
                fontfamily='sans-serif', style='italic')
    return fig, ax


# ─────────────────────────────────────────────────────────────────────────────
# C1 — NEQR POSITION SUPERPOSITION
# ─────────────────────────────────────────────────────────────────────────────
def c1_neqr_position():
    row_labels = [
        '|0⟩ pos_Y[0]','|0⟩ pos_Y[1]','|0⟩ pos_Y[2]','|0⟩ pos_Y[3]','|0⟩ pos_Y[4]',
        '|0⟩ pos_X[0]','|0⟩ pos_X[1]','|0⟩ pos_X[2]','|0⟩ pos_X[3]','|0⟩ pos_X[4]',
    ]
    N = len(row_labels)
    fig, ax = make_ax(N, 10,
        'C1 — NEQR Position Superposition',
        'H on 10 position qubits → all 1024 pixel coordinates active simultaneously')

    for i, lbl in enumerate(row_labels):
        c = TEAL if i < 5 else ORG
        rlabel(ax, i, lbl, color=c)
        wire(ax, i, 0, 9.5, color=c, lw=1.4)

    for i in range(N):
        H(ax, 1.0, i)

    barrier(ax, 1.7, 0, N-1)

    ax.annotate('', xy=(-0.52, 0), xytext=(-0.52, 4),
                arrowprops=dict(arrowstyle='<->', color=TEAL, lw=1.3))
    ax.text(-0.62, 2, '5 Y\nqubits', ha='right', va='center', color=TEAL, fontsize=7)
    ax.annotate('', xy=(-0.52, 5), xytext=(-0.52, 9),
                arrowprops=dict(arrowstyle='<->', color=ORG, lw=1.3))
    ax.text(-0.62, 7, '5 X\nqubits', ha='right', va='center', color=ORG, fontsize=7)

    ax.text(5.5, N/2 - 0.5,
            '|pos⟩ = (1/√1024)  Σ_{y=0}^{31} Σ_{x=0}^{31}  |y⟩|x⟩\n\n'
            'All 1024 pixel positions active in one circuit',
            ha='center', va='center', color=GOLD, fontsize=9,
            bbox=dict(boxstyle='round,pad=0.5', fc='#1a1a2e', ec=GOLD, lw=1.2))

    plt.tight_layout()
    p = os.path.join(OUT_DIR, 'C1_neqr_position.png')
    fig.savefig(p, dpi=150, bbox_inches='tight', facecolor=BG)
    plt.close(); print(f'  Saved {p}'); return p


# ─────────────────────────────────────────────────────────────────────────────
# C2 — NEQR PIXEL ENCODING
# ─────────────────────────────────────────────────────────────────────────────
def c2_neqr_pixel():
    row_labels = [
        'pos_Y[0]', 'pos_Y[1]',
        '|0⟩ int[0]','|0⟩ int[1]','|0⟩ int[2]','|0⟩ int[3]',
        '|0⟩ int[4]','|0⟩ int[5]','|0⟩ int[6]','|0⟩ int[7]',
    ]
    N = len(row_labels)
    fig, ax = make_ax(N, 17,
        'C2 — NEQR Pixel Encoding  (example: pixel y=1, value=165 = 10100101₂)',
        'MCX gates set intensity bits conditioned on pixel position — repeated for all 1024 pixels',
        fig_h=6)

    row_colors = [TEAL, TEAL] + [WIRE]*8
    for i, (lbl, rc) in enumerate(zip(row_labels, row_colors)):
        rlabel(ax, i, lbl, color=rc)
        wire(ax, i, 0, 16.5, color=rc)

    H(ax, 0.8, 0); H(ax, 0.8, 1)
    barrier(ax, 1.5, 0, N-1)
    blabel(ax, 0.8, N+0.15, '① H\nsuperpos', TEAL)

    Xg(ax, 2.3, 1)
    barrier(ax, 2.9, 0, N-1)
    blabel(ax, 2.3, N+0.15, '② Condition\npos_Y=01\n(y=1)', GOLD)

    val = 165
    x_start = 3.5
    for b in range(8):
        bit = (val >> b) & 1
        irow = 2 + b
        xp   = x_start + b * 1.1
        if bit:
            MCX(ax, xp, [0, 1], irow, color=MCX_C)
        else:
            ax.plot(xp, irow, 'x', color=DIM, ms=6, mew=1.2, zorder=3)
    barrier(ax, x_start + 8*1.1 + 0.3, 0, N-1)
    blabel(ax, x_start + 3.5, N+0.15,
           '③ MCX: set int[b]=1 where 165-bit=1\n165=10100101₂ → int[0,2,5,7] set', GOLD)

    x_unflip = x_start + 8*1.1 + 0.8
    Xg(ax, x_unflip, 1)
    barrier(ax, x_unflip + 0.6, 0, N-1)
    blabel(ax, x_unflip, N+0.15, '④ Restore\npos', ORG)

    x_rep = x_unflip + 1.2
    for i in range(N):
        wire(ax, i, x_rep, x_rep+1.5, color=DIM, lw=0.8)
    ax.text(x_rep+0.75, N/2-0.5, '↺ ×1024\npixels', ha='center', va='center',
            color=DIM, fontsize=8,
            bbox=dict(boxstyle='round,pad=0.3', fc=PANEL, ec=DIM, lw=0.7))

    x_meas = x_rep + 2.2
    for i in range(N):
        draw_meas(ax, x_meas, i)
    blabel(ax, x_meas, N+0.15, '⑤\nMeasure', GRN)

    plt.tight_layout()
    p = os.path.join(OUT_DIR, 'C2_neqr_pixel_encoding.png')
    fig.savefig(p, dpi=150, bbox_inches='tight', facecolor=BG)
    plt.close(); print(f'  Saved {p}'); return p


# ─────────────────────────────────────────────────────────────────────────────
# C3 — SHARE GENERATION (Code 1, all 3 shares)
# ─────────────────────────────────────────────────────────────────────────────
def c3_share_generation():
    row_labels = [
        'G1[0]','G1[1]','G1[2]','G1[3]',   # 0-3   G1 LSB
        'G2[0]','G2[1]','G2[2]','G2[3]',   # 4-7   G2 LSB
        'G1[4]','G1[5]','G1[6]','G1[7]',   # 8-11  G1 MSB
        'G2[4]','G2[5]','G2[6]','G2[7]',   # 12-15 G2 MSB
        'C1[4]','C1[5]','C1[6]','C1[7]',   # 16-19 C1 MSB
        'C2[4]','C2[5]','C2[6]','C2[7]',   # 20-23 C2 MSB
        'C3[4]','C3[5]','C3[6]','C3[7]',   # 24-27 C3 MSB
        'S1[0]','S1[1]','S1[2]','S1[3]',   # 28-31 S1 LSB
        'S1[4]','S1[5]','S1[6]','S1[7]',   # 32-35 S1 MSB
        'S2[0]','S2[1]','S2[2]','S2[3]',   # 36-39 S2 LSB
        'S2[4]','S2[5]','S2[6]','S2[7]',   # 40-43 S2 MSB
        'S3[0]','S3[1]','S3[2]','S3[3]',   # 44-47 S3 LSB
        'S3[4]','S3[5]','S3[6]','S3[7]',   # 48-51 S3 MSB
    ]
    N = len(row_labels)

    def rc(lbl):
        if lbl.startswith('G1'): return '#79c0ff'
        if lbl.startswith('G2'): return '#56d364'
        if lbl.startswith('C'):  return ORG
        if lbl.startswith('S1'): return PRP
        if lbl.startswith('S2'): return TEAL
        if lbl.startswith('S3'): return GOLD
        return WIRE

    fig, ax = make_ax(N, 21,
        'C3 — Share Generation: Code 1 Nibble XOR Logic',
        'S1=G1⊕G2(LSB)+C1(MSB)  |  S2=S1⊕G1(MSB)+C2(MSB)  |  S3=S1⊕S2⊕G2(MSB)+C3(MSB)',
        fig_h=max(10, N*0.42+2))

    G1L=list(range(0,4));  G2L=list(range(4,8))
    G1M=list(range(8,12)); G2M=list(range(12,16))
    C1M=list(range(16,20)); C2M=list(range(20,24)); C3M=list(range(24,28))
    S1L=list(range(28,32)); S1M=list(range(32,36))
    S2L=list(range(36,40)); S2M=list(range(40,44))
    S3L=list(range(44,48)); S3M=list(range(48,52))

    for i, lbl in enumerate(row_labels):
        rlabel(ax, i, lbl, color=rc(lbl))
        wire(ax, i, 0, 20.5, color=rc(lbl), lw=1.2)

    # ── S1[LSB] = G1[LSB] ⊕ G2[LSB] ─────────────────────────────────────────
    x = 1.2
    for b in range(4):
        CNOT(ax, x + b*0.65,        G1L[b], S1L[b], color='#79c0ff')
        CNOT(ax, x + b*0.65 + 0.32, G2L[b], S1L[b], color='#56d364')
    barrier(ax, x+3.1, 0, N-1)
    blabel(ax, x+1.3, N+0.12, 'S1[LSB] = G1[LSB] ⊕ G2[LSB]', '#79c0ff')

    # ── S1[MSB] = C1[MSB] ────────────────────────────────────────────────────
    x = 4.8
    for b in range(4):
        CNOT(ax, x + b*0.65, C1M[b], S1M[b], color=ORG)
    barrier(ax, x+3.1, 0, N-1)
    blabel(ax, x+1.3, N+0.12, 'S1[MSB] = C1[MSB]', ORG)

    # ── S2[LSB] = S1[MSB] ⊕ G1[MSB] ─────────────────────────────────────────
    x = 8.4
    for b in range(4):
        CNOT(ax, x + b*0.65,        S1M[b], S2L[b], color=PRP)
        CNOT(ax, x + b*0.65 + 0.32, G1M[b], S2L[b], color='#79c0ff')
    barrier(ax, x+3.1, 0, N-1)
    blabel(ax, x+1.3, N+0.12, 'S2[LSB] = S1[MSB] ⊕ G1[MSB]', PRP)

    # ── S2[MSB] = C2[MSB] ────────────────────────────────────────────────────
    x = 12.0
    for b in range(4):
        CNOT(ax, x + b*0.65, C2M[b], S2M[b], color=ORG)
    barrier(ax, x+3.1, 0, N-1)
    blabel(ax, x+1.3, N+0.12, 'S2[MSB] = C2[MSB]', ORG)

    # ── S3[LSB] = S1[MSB] ⊕ S2[MSB] ⊕ G2[MSB] ───────────────────────────────
    x = 15.6
    for b in range(4):
        CNOT(ax, x + b*0.55,        S1M[b], S3L[b], color=PRP)
        CNOT(ax, x + b*0.55 + 0.2,  S2M[b], S3L[b], color=TEAL)
        CNOT(ax, x + b*0.55 + 0.4,  G2M[b], S3L[b], color='#56d364')
    barrier(ax, x+2.8, 0, N-1)
    blabel(ax, x+1.2, N+0.12, 'S3[LSB] = S1[MSB] ⊕ S2[MSB] ⊕ G2[MSB]', TEAL)

    # ── S3[MSB] = C3[MSB] ────────────────────────────────────────────────────
    x = 19.0
    for b in range(4):
        CNOT(ax, x + b*0.35, C3M[b], S3M[b], color=ORG)
    blabel(ax, x+0.7, N+0.12, 'S3[MSB] = C3[MSB]', ORG)

    plt.tight_layout()
    p = os.path.join(OUT_DIR, 'C3_share_generation.png')
    fig.savefig(p, dpi=150, bbox_inches='tight', facecolor=BG)
    plt.close(); print(f'  Saved {p}'); return p


# ─────────────────────────────────────────────────────────────────────────────
# C4 — QPA: QUANTUM PARITY AUTHENTICATION
# ─────────────────────────────────────────────────────────────────────────────
def c4_qpa():
    row_labels = ['S[0]','S[1]','S[2]','S[3]','S[4]','S[5]','S[6]','S[7]', '|0⟩ qpa']
    N = len(row_labels)
    fig, ax = make_ax(N, 17,
        'C4 — Quantum Parity Authentication (QPA)',
        'qpa = S[0]⊕S[1]⊕...⊕S[7]  |  Measured same shot as share  |  Deterministic: 100% accuracy')

    qpa_row = 8
    for i, lbl in enumerate(row_labels):
        c = QPA_C if i == qpa_row else WIRE
        rlabel(ax, i, lbl, color=c)
        wire(ax, i, 0, 16.5, color=c)

    barrier(ax, 0.8, 0, N-1)
    blabel(ax, 0.8, N+0.15, 'Share S\nNEQR-encoded', DIM)

    # 8 CNOT: S[b] → qpa
    for b in range(8):
        xp = 1.8 + b * 1.25
        CNOT(ax, xp, b, qpa_row, color=QPA_C)

    barrier(ax, 11.8, 0, N-1)
    blabel(ax, 6.5, N+0.15, 'qpa = S[0] ⊕ S[1] ⊕ S[2] ⊕ S[3] ⊕ S[4] ⊕ S[5] ⊕ S[6] ⊕ S[7]', QPA_C)

    for i in range(N):
        MEAS(ax, 12.7, i)
    blabel(ax, 12.7, N+0.15, 'Measure\nsame shot', GRN)

    for i in range(N):
        cwire(ax, i, 13.2, 14.8)

    # Proof box
    ax.text(15.5, 3.8,
            'PER SHOT:\nmeas(qpa)\n  ==\nparity(\n  meas(S))\n\nALWAYS ✓\n100%',
            ha='center', va='center', color=GRN, fontsize=9, fontfamily='monospace',
            bbox=dict(boxstyle='round,pad=0.5', fc='#0d2a0d', ec=GRN, lw=1.5))

    # Outcome table
    ax.text(15.5, 7.8,
            'meas(S) even → qpa=0\nmeas(S) odd  → qpa=1',
            ha='center', va='center', color=GOLD, fontsize=8, fontfamily='monospace',
            bbox=dict(boxstyle='round,pad=0.4', fc='#2a1a00', ec=GOLD, lw=1))

    # At decode: annotation
    ax.text(3.5, -0.62,
            'At DECODE TIME: same circuit run on loaded shares → '
            'parity_decode[y,x] compared vs parity_encode[y,x]\n'
            'Match=authentic  |  Mismatch=tampered pixel at exact location',
            ha='center', va='center', color=TEXT, fontsize=8, fontfamily='sans-serif',
            bbox=dict(boxstyle='round,pad=0.4', fc='#1a1a1a', ec=QPA_C, lw=1))

    plt.tight_layout()
    p = os.path.join(OUT_DIR, 'C4_qpa.png')
    fig.savefig(p, dpi=150, bbox_inches='tight', facecolor=BG)
    plt.close(); print(f'  Saved {p}'); return p


# ─────────────────────────────────────────────────────────────────────────────
# C5 — FULL ENCODING PIPELINE
# ─────────────────────────────────────────────────────────────────────────────
def c5_full_encoding():
    row_labels = [
        'pos[0..9]',
        'G1[0..7]', 'G2[0..7]',
        'C1[0..7]', 'C2[0..7]', 'C3[0..7]',
        'S1[0..7]', 'S2[0..7]', 'S3[0..7]',
        '|0⟩ qpa1', '|0⟩ qpa2', '|0⟩ qpa3',
    ]
    N = len(row_labels)
    rc = [BLUE, '#79c0ff','#56d364', ORG,ORG,ORG, PRP,TEAL,GOLD, QPA_C,QPA_C,QPA_C]

    fig, ax = make_ax(N, 23,
        'C5 — Full Encoding Pipeline (all stages)',
        '① H superposition  ② NEQR encode  ③ Code 1 CNOT chains  ④ QPA parity  ⑤ Measure',
        fig_h=7)

    for i, (lbl, c) in enumerate(zip(row_labels, rc)):
        rlabel(ax, i, lbl, color=c)
        wire(ax, i, 0, 22.5, color=c, lw=1.4)

    # ① H on pos
    H(ax, 0.8, 0)
    barrier(ax, 1.5, 0, N-1)
    blabel(ax, 0.8, N+0.12, '①\nH: 1024\npositions', TEAL)

    # ② NEQR encode (shown as labelled boxes)
    for i in range(1, 6):
        gate(ax, 2.8, i, 'NEQR', w=0.9, h=0.30, fc='#0d2137', tc=rc[i], fs=7, bold=False)
        ax.plot([2.8, 2.8], [0.16, i-0.16], color=MCX_C, lw=0.7, ls=':', zorder=2)
    ax.add_patch(Circle((2.8, 0), 0.06, fc=CTRL, ec=CTRL, zorder=5))
    barrier(ax, 3.5, 0, N-1)
    blabel(ax, 2.8, N+0.12, '②\nNEQR encode\nG1,G2,C1,C2,C3', GOLD)

    # ③ Code 1 share gen — CNOT chains (simplified)
    CNOT(ax, 4.4,  1, 6, color='#79c0ff')
    CNOT(ax, 4.7,  2, 6, color='#56d364')
    CNOT(ax, 5.0,  3, 6, color=ORG)
    CNOT(ax, 5.4,  6, 7, color=PRP)
    CNOT(ax, 5.7,  1, 7, color='#79c0ff')
    CNOT(ax, 6.0,  4, 7, color=ORG)
    CNOT(ax, 6.4,  6, 8, color=PRP)
    CNOT(ax, 6.7,  7, 8, color=TEAL)
    CNOT(ax, 7.0,  2, 8, color='#56d364')
    CNOT(ax, 7.3,  5, 8, color=ORG)
    barrier(ax, 8.0, 0, N-1)
    blabel(ax, 6.2, N+0.12, '③\nCode 1 CNOT chains\nS1, S2, S3 generated', PRP)

    # ④ QPA: 8 CNOT per share
    for s, (s_row, qpa_row) in enumerate([(6,9),(7,10),(8,11)]):
        for b in range(8):
            xp = 8.8 + s*0.22 + b*0.20
            CNOT(ax, xp, s_row, qpa_row, color=QPA_C)
    barrier(ax, 11.0, 0, N-1)
    blabel(ax, 9.8, N+0.12, '④ QPA: 8 CNOT × 3 shares = 24 gates\nqpa_i = parity(S_i)', QPA_C)

    # ⑤ Measure
    for i in range(N):
        MEAS(ax, 12.0, i)
    for i in range(N):
        cwire(ax, i, 12.5, 14.2)
    barrier(ax, 11.5, 0, N-1)
    blabel(ax, 12.0, N+0.12, '⑤\nMeasure\nall in\nsame shot', GRN)

    # Output labels
    out_map = {0:'pos(y,x)', 6:'S1 value',7:'S2 value',8:'S3 value',
               9:'qpa1 bit',10:'qpa2 bit',11:'qpa3 bit'}
    for row, lbl in out_map.items():
        ax.text(15.5, row, f'→ {lbl}', va='center',
                color=rc[row], fontsize=7, fontfamily='monospace')

    ax.text(19.5, 5.5,
            '8192 shots total:\n'
            '• Share images built\n'
            '  (majority vote/pixel)\n'
            '• Parity maps saved\n'
            '  (1 bit/pixel/share)\n'
            '• QPA accuracy: 100%\n'
            '  (8192/8192 shots)\n'
            '• 77 qubits, 225k gates',
            ha='left', va='center', color=TEXT, fontsize=8, fontfamily='sans-serif',
            bbox=dict(boxstyle='round,pad=0.5', fc='#1a1a1a', ec=DIM, lw=0.8))

    plt.tight_layout()
    p = os.path.join(OUT_DIR, 'C5_full_encoding.png')
    fig.savefig(p, dpi=150, bbox_inches='tight', facecolor=BG)
    plt.close(); print(f'  Saved {p}'); return p


# ─────────────────────────────────────────────────────────────────────────────
# C6 — DECODING + QPA VERIFICATION
# ─────────────────────────────────────────────────────────────────────────────
def c6_decoding():
    row_labels = [
        'pos[0..9]',
        'S1[0..7]', 'S2[0..7]', 'S3[0..7]',
        '|0⟩ qpa1', '|0⟩ qpa2', '|0⟩ qpa3',
        '|0⟩ R1[0..7]', '|0⟩ R2[0..7]',
    ]
    N = len(row_labels)
    rc = [BLUE, PRP, TEAL, GOLD, QPA_C, QPA_C, QPA_C, '#79c0ff', '#56d364']

    fig, ax = make_ax(N, 23,
        'C6 — Decoding: Reverse CNOT + QPA Tamper Verification',
        '① H pos  ② NEQR encode shares  ③ QPA verify  ④ Reverse Code1 → recover G1,G2  ⑤ Measure',
        fig_h=6.5)

    for i, (lbl, c) in enumerate(zip(row_labels, rc)):
        rlabel(ax, i, lbl, color=c)
        wire(ax, i, 0, 22.5, color=c, lw=1.4)

    # ① H
    H(ax, 0.8, 0)
    barrier(ax, 1.5, 0, N-1)
    blabel(ax, 0.8, N+0.12, '①\nH: pos', TEAL)

    # ② NEQR encode shares
    for i in range(1, 4):
        gate(ax, 2.6, i, 'NEQR', w=0.9, h=0.30, fc='#0d2137', tc=rc[i], fs=7, bold=False)
        ax.plot([2.6, 2.6], [0.16, i-0.16], color=MCX_C, lw=0.7, ls=':', zorder=2)
    ax.add_patch(Circle((2.6, 0), 0.06, fc=CTRL, ec=CTRL, zorder=5))
    barrier(ax, 3.3, 0, N-1)
    blabel(ax, 2.6, N+0.12, '②\nNEQR encode\nS1, S2, S3', BLUE)

    # ③ QPA (same as encoding)
    for s, (s_row, qpa_row) in enumerate([(1,4),(2,5),(3,6)]):
        for b in range(8):
            xp = 3.8 + s*0.18 + b*0.20
            CNOT(ax, xp, s_row, qpa_row, color=QPA_C)
    for qr in [4,5,6]:
        MEAS(ax, 5.6, qr)
    for qr in [4,5,6]:
        cwire(ax, qr, 6.1, 7.5)
    barrier(ax, 6.0, 0, N-1)
    blabel(ax, 4.8, N+0.12,
           '③ QPA: parity_decode vs parity_encode\n→ 0 mismatch=AUTHENTIC  |  mismatch=TAMPERED', QPA_C)

    # ④ Reverse Code 1
    # R1 = G1 recovered: S1[MSB] XOR S2[LSB]
    for b in range(4):
        CNOT(ax, 7.0+b*0.45, 1, 7, color=PRP)
        CNOT(ax, 7.2+b*0.45, 2, 7, color=TEAL)
    # R2 = G2 recovered: S1[MSB] XOR S2[MSB] XOR S3[LSB]
    for b in range(4):
        CNOT(ax, 9.0+b*0.38, 1, 8, color=PRP)
        CNOT(ax, 9.15+b*0.38, 2, 8, color=TEAL)
        CNOT(ax, 9.3+b*0.38, 3, 8, color=GOLD)
    barrier(ax, 11.0, 0, N-1)
    blabel(ax, 9.0, N+0.12,
           '④ Reverse Code1\nR1=S1[MSB]⊕S2[LSB]→G1  R2=S1⊕S2⊕S3(MSBs)→G2', GOLD)

    # ⑤ Measure
    for i in [0,1,2,3,7,8]:
        MEAS(ax, 12.0, i)
    for i in [0,1,2,3,7,8]:
        cwire(ax, i, 12.5, 14.0)
    blabel(ax, 12.0, N+0.12, '⑤\nMeasure', GRN)

    # QPA result box
    ax.text(16.0, 2.5,
            'QPA Result:\n'
            '──────────────────\n'
            'Untampered:\n'
            '  parity_dec = parity_enc\n'
            '  1024/1024 match\n'
            '  → AUTHENTIC ✓\n\n'
            'Tampered (30% pixels):\n'
            '  ~700/1024 match\n'
            '  326 pixels detected\n'
            '  → TAMPERED ✗',
            ha='left', va='center', color=TEXT, fontsize=8, fontfamily='monospace',
            bbox=dict(boxstyle='round,pad=0.5', fc='#0d150d', ec=QPA_C, lw=1.2))

    ax.text(16.0, 7.2,
            'Recovery:\n'
            '──────────────────\n'
            'G1: PSNR=38.59 dB\n'
            '    SSIM=0.9984\n'
            'G2: PSNR=34.15 dB\n'
            '    SSIM=0.9963',
            ha='left', va='center', color=TEXT, fontsize=8, fontfamily='monospace',
            bbox=dict(boxstyle='round,pad=0.5', fc='#0d0d15', ec=BLUE, lw=1.2))

    plt.tight_layout()
    p = os.path.join(OUT_DIR, 'C6_decoding.png')
    fig.savefig(p, dpi=150, bbox_inches='tight', facecolor=BG)
    plt.close(); print(f'  Saved {p}'); return p

# ─────────────────────────────────────────────────────────────────────────────
# C7 — EBA: ENTANGLEMENT-BASED AUTHENTICATION
# ─────────────────────────────────────────────────────────────────────────────
def c7_eba():
    # EBA focuses strictly on the MSB and the two new auxiliary qubits
    row_labels = ['S[7] (MSB)', '|0⟩ anc', '|0⟩ tag']
    N = len(row_labels)
    
    # We use RED to symbolize the "Alarm" tripwire nature of EBA
    EBA_C = '#f85149' 
    
    fig, ax = make_ax(N, 16,
        'C7 — Entanglement-Based Authentication (EBA)',
        'Bell state (|Φ+⟩) entangled with MSB to detect measurement/intercept-resend attacks',
        fig_h=4.5)

    for i, lbl in enumerate(row_labels):
        # Color the MSB as standard wire, but the EBA qubits as alarm-red
        c = EBA_C if i > 0 else WIRE
        rlabel(ax, i, lbl, color=c)
        wire(ax, i, 0, 15.5, color=c)

    barrier(ax, 0.8, 0, N-1)
    blabel(ax, 0.8, N+0.15, 'Share MSB\nencoded', DIM)

    # 1. Hadamard on ancilla
    H(ax, 2.0, 1)

    # 2. CNOT anc -> tag (Creates the Bell pair)
    CNOT(ax, 3.5, 1, 2, color=EBA_C)
    barrier(ax, 4.5, 0, N-1)
    blabel(ax, 2.75, N+0.15, 'Bell State\n|Φ+⟩ Created', TEAL)

    # 3. CNOT MSB -> tag (Entangles the share with the Bell state)
    # This is the "Lock" binding the share to the alarm
    CNOT(ax, 6.0, 0, 2, color=PRP)
    barrier(ax, 7.5, 0, N-1)
    blabel(ax, 6.0, N+0.15, 'Entangle MSB\nwith Tag', PRP)

    # 4. Measure all three qubits
    for i in range(N):
        MEAS(ax, 9.0, i)
    blabel(ax, 9.0, N+0.15, 'Measure\nfor Auth', GRN)

    for i in range(N):
        cwire(ax, i, 9.5, 11.0)

    # 5. Verification Logic Box (Shows the math behind the alarm)
    ax.text(13.2, 1.0,
            'VERIFICATION LOGIC:\n'
            '───────────────────\n'
            'If meas(MSB) == 0:\n'
            '  anc == tag (100% Correlated)\n\n'
            'If meas(MSB) == 1:\n'
            '  anc != tag (100% Anti-corr.)\n\n'
            'If Intercepted / Measured:\n'
            '  Entanglement snaps.\n'
            '  Correlation drops to ~50%',
            ha='center', va='center', color=TEXT, fontsize=9, fontfamily='monospace',
            bbox=dict(boxstyle='round,pad=0.5', fc='#1a0a0a', ec=EBA_C, lw=1.5))

    plt.tight_layout()
    p = os.path.join(OUT_DIR, 'C7_eba.png')
    fig.savefig(p, dpi=150, bbox_inches='tight', facecolor=BG)
    plt.close(); print(f'  Saved {p}'); return p
# ─────────────────────────────────────────────────────────────────────────────
# MASTER OVERVIEW
# ─────────────────────────────────────────────────────────────────────────────
def master_overview():
    fig = plt.figure(figsize=(22, 14))
    fig.patch.set_facecolor(BG)
    gs = gridspec.GridSpec(2, 1, figure=fig, hspace=0.08)

    def bx(ax, x, y, w, h, text, sub='', fc=BLUE, tc=TEXT, fs=9):
        r = FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.06',
                           fc=fc, ec=tc, lw=1.2, alpha=0.9)
        ax.add_patch(r)
        ax.text(x+w/2, y+h/2+(0.07 if sub else 0), text,
                ha='center', va='center', color=tc,
                fontsize=fs, fontweight='bold', fontfamily='sans-serif')
        if sub:
            ax.text(x+w/2, y+h/2-0.14, sub,
                    ha='center', va='center', color=tc,
                    fontsize=fs-2, fontfamily='sans-serif', style='italic')

    def arr(ax, x1, x2, y=2.4):
        ax.annotate('', xy=(x2, y), xytext=(x1, y),
                    arrowprops=dict(arrowstyle='->', color=GOLD, lw=2.2))

    # ── ENCODING ROW ─────────────────────────────────────────────────────────
    ax1 = fig.add_subplot(gs[0])
    ax1.set_facecolor(BG); ax1.set_xlim(0,22); ax1.set_ylim(0,5.2); ax1.axis('off')
    ax1.text(11,5.0,'ENCODING PIPELINE',ha='center',color=GOLD,fontsize=13,fontweight='bold')

    # S1
    bx(ax1,0.1,3.5,2.1,0.8,'Secrets G1, G2','32×32 grayscale',GRN,'#0d1117')
    bx(ax1,0.1,2.5,2.1,0.8,'Covers C1,C2,C3','32×32 grayscale','#0d6060',TEXT)
    bx(ax1,0.1,1.5,2.1,0.8,'|pos⟩ H⊗10','1024 pos simultaneous',BLUE,TEXT)
    bx(ax1,0.1,0.6,2.1,0.8,'77 qubits total','225,585 gates',DIM,TEXT,fs=8)
    ax1.text(1.15,0.1,'STAGE 1\nInputs',ha='center',color=GOLD,fontsize=8,fontweight='bold')
    arr(ax1,2.2,2.9)

    # S2
    bx(ax1,2.9,3.0,2.8,1.3,'NEQR Encoding','MCX per pixel\n|f(y,x)⟩|y⟩|x⟩','#0d2137',BLUE)
    bx(ax1,2.9,1.5,2.8,1.3,'5 images encoded','G1,G2,C1,C2,C3\n8 int qubits each','#0d2137',TEAL)
    bx(ax1,2.9,0.5,2.8,0.8,'depth=61,243','Hadamard + MCX gates',DIM,TEXT,fs=7)
    ax1.text(4.3,0.1,'STAGE 2\nNEQR Encode',ha='center',color=GOLD,fontsize=8,fontweight='bold')
    arr(ax1,5.7,6.4)

    # S3
    bx(ax1,6.4,3.9,3.3,0.65,'S1[LSB] = G1[LSB] ⊕ G2[LSB]','','#2a1a5c',PRP,fs=8)
    bx(ax1,6.4,3.2,3.3,0.65,'S1[MSB] = C1[MSB]','','#2a1a5c',ORG,fs=8)
    bx(ax1,6.4,2.5,3.3,0.65,'S2[LSB] = S1[MSB] ⊕ G1[MSB]','','#1a2a5c',TEAL,fs=8)
    bx(ax1,6.4,1.8,3.3,0.65,'S2[MSB] = C2[MSB]','','#1a2a5c',ORG,fs=8)
    bx(ax1,6.4,1.1,3.3,0.65,'S3[LSB] = S1⊕S2⊕G2 (MSBs)','','#1a3a2c',GOLD,fs=7)
    bx(ax1,6.4,0.4,3.3,0.65,'S3[MSB] = C3[MSB]','CNOT chains','#1a3a2c',ORG,fs=8)
    ax1.text(8.05,0.1,'STAGE 3\nCode 1 CNOT',ha='center',color=GOLD,fontsize=8,fontweight='bold')
    arr(ax1,9.7,10.4)

    # S4
    bx(ax1,10.4,3.5,3.0,0.8,'QPA: qpa1=parity(S1)','S1[0]⊕...⊕S1[7]',QPA_C,TEXT,fs=8)
    bx(ax1,10.4,2.6,3.0,0.8,'QPA: qpa2=parity(S2)','S2[0]⊕...⊕S2[7]',QPA_C,TEXT,fs=8)
    bx(ax1,10.4,1.7,3.0,0.8,'QPA: qpa3=parity(S3)','S3[0]⊕...⊕S3[7]',QPA_C,TEXT,fs=8)
    bx(ax1,10.4,0.5,3.0,1.0,'+3 qubits, +24 gates\n100% deterministic','per-pixel auth bit',DIM,GOLD,fs=8)
    ax1.text(11.9,0.1,'STAGE 4\nQPA Auth',ha='center',color=GOLD,fontsize=8,fontweight='bold')
    arr(ax1,13.4,14.1)

    # S5
    bx(ax1,14.1,3.4,2.0,0.8,'Measure pos','→ pixel (y,x)',MEAS_C,TEXT,fs=8)
    bx(ax1,14.1,2.5,2.0,0.8,'Measure S1,S2,S3','→ share values',MEAS_C,TEXT,fs=8)
    bx(ax1,14.1,1.6,2.0,0.8,'Measure qpa1,2,3','→ parity bits',MEAS_C,TEXT,fs=8)
    bx(ax1,14.1,0.5,2.0,0.9,'8192 shots\nmajority vote','per pixel',DIM,TEXT,fs=8)
    ax1.text(15.1,0.1,'STAGE 5\nMeasure',ha='center',color=GOLD,fontsize=8,fontweight='bold')
    arr(ax1,16.1,16.7)

    # Output
    bx(ax1,16.7,3.4,2.2,0.75,'S1.npy, S2.npy, S3.npy','Share images',GRN,'#0d1117',fs=7)
    bx(ax1,16.7,2.55,2.2,0.75,'parity_S1/S2/S3.npy','QPA maps',QPA_C,'#0d1117',fs=7)
    bx(ax1,16.7,1.7,2.2,0.75,'Entropy~7.2 bits','Share randomness',DIM,TEXT,fs=7)
    bx(ax1,16.7,0.5,2.2,1.0,'825s encoding\n~13.8 min','MQT DDSIM',DIM,TEXT,fs=7)
    ax1.text(17.8,0.1,'OUTPUT',ha='center',color=GRN,fontsize=8,fontweight='bold')

    # ── DECODING ROW ─────────────────────────────────────────────────────────
    ax2 = fig.add_subplot(gs[1])
    ax2.set_facecolor(BG); ax2.set_xlim(0,22); ax2.set_ylim(0,5.2); ax2.axis('off')
    ax2.text(11,5.0,'DECODING PIPELINE + QPA TAMPER DETECTION',
             ha='center',color=TEAL,fontsize=13,fontweight='bold')

    # D1
    bx(ax2,0.1,3.5,2.1,0.8,'Load S1,S2,S3','.npy files',BLUE,TEXT,fs=8)
    bx(ax2,0.1,2.5,2.1,0.8,'Load parity maps','P1,P2,P3 reference',QPA_C,TEXT,fs=8)
    bx(ax2,0.1,1.5,2.1,0.8,'|pos⟩ H⊗10','superposition ready',PRP,TEXT,fs=8)
    bx(ax2,0.1,0.5,2.1,0.8,'37 qubits','QPA circuit',DIM,TEXT,fs=8)
    ax2.text(1.15,0.1,'STAGE 1\nLoad',ha='center',color=TEAL,fontsize=8,fontweight='bold')
    arr(ax2,2.2,2.9)

    # D2
    bx(ax2,2.9,4.0,3.0,0.55,'NEQR encode S1,S2,S3','same as encoding',BLUE,TEXT,fs=7)
    bx(ax2,2.9,3.2,3.0,0.75,'QPA (identical circuit)','qpa_dec_i=Si[0]⊕...⊕Si[7]',QPA_C,TEXT,fs=8)
    bx(ax2,2.9,2.1,3.0,1.0,'Compare parity maps\ndec[y,x] vs enc[y,x]','','#0a2a0a',GRN,fs=8)
    bx(ax2,2.9,0.4,3.0,1.5,
       'Untampered: 1024/1024\nTampered: ~700/1024\n→ exact pixels flagged',
       '','#0a1a0a',GRN,fs=7)
    ax2.text(4.4,0.1,'STAGE 2\nQPA Verify',ha='center',color=TEAL,fontsize=8,fontweight='bold')
    arr(ax2,5.9,6.6)

    # D3
    bx(ax2,6.6,3.7,3.2,0.65,'R1[LSB]=S1[MSB]⊕S2[LSB]','→ G1 MSB nibble','#2a2a0a',GOLD,fs=7)
    bx(ax2,6.6,2.95,3.2,0.65,'R1[MSB]=S1[LSB]','→ G1⊕G2 combined LSB','#2a2a0a',GOLD,fs=7)
    bx(ax2,6.6,2.2,3.2,0.65,'R2[LSB]=S1⊕S2⊕S3 (MSBs)','→ G2 MSB nibble','#2a2a0a',GOLD,fs=7)
    bx(ax2,6.6,1.5,3.2,0.65,'R2[MSB]=S3[MSB]','→ C3 MSB verify','#2a2a0a',TEXT,fs=7)
    bx(ax2,6.6,0.4,3.2,0.9,'Exact inverse of encoding\nCNOT chain reversed',
       '','#1a1a0a',ORG,fs=7)
    ax2.text(8.2,0.1,'STAGE 3\nReverse CNOT',ha='center',color=TEAL,fontsize=8,fontweight='bold')
    arr(ax2,9.8,10.5)

    # D4
    bx(ax2,10.5,3.3,2.0,0.8,'Measure pos','→ pixel (y,x)',MEAS_C,TEXT,fs=8)
    bx(ax2,10.5,2.4,2.0,0.8,'Measure R1','→ G1 value',MEAS_C,TEXT,fs=8)
    bx(ax2,10.5,1.5,2.0,0.8,'Measure R2','→ G2 value',MEAS_C,TEXT,fs=8)
    bx(ax2,10.5,0.5,2.0,0.8,'8192 shots','majority vote',DIM,TEXT,fs=8)
    ax2.text(11.5,0.1,'STAGE 4\nMeasure',ha='center',color=TEAL,fontsize=8,fontweight='bold')
    arr(ax2,12.5,13.2)

    # D5
    bx(ax2,13.2,3.4,2.2,0.75,'G1 Recovered','PSNR=38.59dB SSIM=0.9984',GRN,'#0d1117',fs=7)
    bx(ax2,13.2,2.55,2.2,0.75,'G2 Recovered','PSNR=34.15dB SSIM=0.9963',GRN,'#0d1117',fs=7)
    bx(ax2,13.2,1.7,2.2,0.75,'50 qubits','~115s execution',DIM,TEXT,fs=7)
    bx(ax2,13.2,0.5,2.2,1.0,'QPA: 0 mismatch\n= AUTHENTIC','1024/1024 match',QPA_C,'#0d1117',fs=7)
    ax2.text(14.3,0.1,'OUTPUT',ha='center',color=GRN,fontsize=8,fontweight='bold')

    # Highlight
    bx(ax2,16.0,0.3,5.7,4.5,
       'QPA Tamper Detection Summary:\n\n'
       '  Per-shot:  meas(qpa) = parity(meas(S))  [100%]\n\n'
       '  Untampered:  parity_dec = parity_enc\n'
       '               1024/1024 match → AUTHENTIC\n\n'
       '  Tampered (30% pixels, MSB flip):\n'
       '               ~700/1024 match\n'
       '               326/308/304 pixels detected\n'
       '               Exact locations on heatmap\n\n'
       '  8 CNOT gates per share  ×  3 shares\n'
       '  = 24 extra gates  |  +3 qubits only',
       '','#0a0a1a',QPA_C,fs=8)

    fig.suptitle(
        'NEQR-Based Quantum Secret Sharing with Quantum Parity Authentication (QPA)\n'
        'Full Scheme: Encoding → QPA Authentication → Decoding → Tamper Detection',
        color=TEXT, fontsize=14, fontweight='bold', y=0.997
    )

    p = os.path.join(OUT_DIR, 'MASTER_overview.png')
    fig.savefig(p, dpi=150, bbox_inches='tight', facecolor=BG)
    plt.close(); print(f'  Saved {p}'); return p


# ─────────────────────────────────────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────────────────────────────────────
def main():
    print('='*70)
    print('CONCEPTUAL QUANTUM CIRCUITS — NEQR + Code 1 + QPA + EBA')
    print('='*70)
    paths = []
    print('\n[C1] NEQR Position Superposition');   paths.append(c1_neqr_position())
    print('\n[C2] NEQR Pixel Encoding');            paths.append(c2_neqr_pixel())
    print('\n[C3] Share Generation (Code 1)');      paths.append(c3_share_generation())
    print('\n[C4] Quantum Parity Authentication');  paths.append(c4_qpa())
    print('\n[C5] Full Encoding Pipeline');         paths.append(c5_full_encoding())
    print('\n[C6] Decoding + QPA Verification');    paths.append(c6_decoding())
    
    # ADD THE NEW EBA CALL HERE
    print('\n[C7] Entanglement-Based Auth (EBA)');  paths.append(c7_eba())
    
    print('\n[Master] Full Scheme Overview');       paths.append(master_overview())

    print('\n'+'='*70)
    print(f'All {len(paths)} diagrams saved to: {OUT_DIR}/')
    print('='*70)
if __name__ == '__main__':
    main()

CONCEPTUAL QUANTUM CIRCUITS — NEQR + Code 1 + QPA + EBA

[C1] NEQR Position Superposition
  Saved conceptual_circuits\C1_neqr_position.png

[C2] NEQR Pixel Encoding
  Saved conceptual_circuits\C2_neqr_pixel_encoding.png

[C3] Share Generation (Code 1)
  Saved conceptual_circuits\C3_share_generation.png

[C4] Quantum Parity Authentication
  Saved conceptual_circuits\C4_qpa.png

[C5] Full Encoding Pipeline
  Saved conceptual_circuits\C5_full_encoding.png

[C6] Decoding + QPA Verification
  Saved conceptual_circuits\C6_decoding.png

[C7] Entanglement-Based Auth (EBA)
  Saved conceptual_circuits\C7_eba.png

[Master] Full Scheme Overview
  Saved conceptual_circuits\MASTER_overview.png

All 8 diagrams saved to: conceptual_circuits/
